In [ ]:
import pathlib
import random
import copy
import numpy as np
import torch

from captum.attr import *
import matplotlib.pyplot as plt
from omegaconf import OmegaConf
import yaml
import argparse
from data.data_loader import load_eeg_data, get_sliding_window_data, create_dataloader
import os
import mne
from tqdm import tqdm
from sklearn.preprocessing import MinMaxScaler
import pandas as pd

from scipy.cluster.hierarchy import fcluster

from models.s4net import S4PatchedFinalNet,TrunkNet, HeadNet

In [ ]:
mne.set_log_level("Error")

In [ ]:
CFG_YAML = """
wandb:
 key: f0c92a0059bf12e2647f0a1c22fdcd12555fa6df
model:
dataset:
 data_directory: /home/marco/Documents/GitHub/tms_eeg_decoding/data
 #file_name: subject_{:03d}_preprocessed_combined_py.fif
 file_name: subject_{:03d}_preprocessed_combined_py.fif
 exclude_timepoints: 100
 subject_index: 1
 test_subject_indices: [1,2,13,24,26,27,29,34,35,41, 42,43,45,46,47,48,52,55,56,57,60,62,67,69,72,73,79,80,86,88,92,102]
 #test_subject_indices: [2]
training:
 training_start_len: 100
 pretrain_epochs: 100
 pretrain_lr: 0.0001
 val_window_len: 1
 epochs_per_window: 10
 num_warmup_epochs: 5
 num_epochs: 800
 slide_step: 1
 num_warmup_epochs_per_window: 0
 lr: 0.005 #maybe change back to 0.0001
 nll_beta: 0.001
 num_warmup_epochs: 0
 batch_size: 50 #better to use 50
 random_seed: 42
 precision: bf16
 kde_lambda: 0.5
 finetune_entire_model: true # Set to true to finetune the entire model, false for transformer only
exp_name: S4_S4EEGNet_ema
"""

def load_config():
    cfg = OmegaConf.create(yaml.safe_load(CFG_YAML))
    cfg.exp_name = f"{cfg.exp_name}_subject_{cfg.dataset.subject_index}"
    return cfg

def parse_args():
    parser = argparse.ArgumentParser()
    parser.add_argument("--update_conf", nargs="*", help="Updates to the configuration in the form of key=value pairs", default=[])
    parser.add_argument("-f", "--fff", help="A dummy argument to handle IPython's default argument", default="1")
    return parser.parse_args()

def update_config(cfg, cli_args):
    for update in cli_args.update_conf:
        key, value = update.split("=")
        try:
            value = eval(value)
        except:
            pass
        OmegaConf.update(cfg, key, value, force_add=True)
    cfg.exp_name = cfg.exp_name + "_" + "_".join(cli_args.update_conf)
    print(OmegaConf.to_yaml(cfg))
    return cfg


def save_config(cfg):
    os.makedirs("conf/sweeps", exist_ok=True)
    os.makedirs("exp/withinsubs", exist_ok=True)
    with open(f"conf/sweeps/withinsubs_{cfg.exp_name}.yaml", "w") as f:
        f.write(OmegaConf.to_yaml(cfg))

In [ ]:
import matplotlib.pylab as pylab
params = {'legend.fontsize': 'x-large',
          'figure.titlesize': 'x-large',
          'figure.figsize': (15, 5),
         'axes.labelsize': 'x-large',
         'axes.titlesize':'x-large',
         'xtick.labelsize':'x-large',
         'ytick.labelsize':'x-large'}
pylab.rcParams.update(params)

In [ ]:
import pickle
import pickle

def load_predicted_amplitude_for_subject(subject_index=2, rep=1):
    data_dir = f"/home/marco/Documents/GitHub/tms_eeg_decoding/subject_interpretability/gradshap_explanations_rep_{rep}"
    file_path = os.path.join(data_dir, f"gradshap_data_subject_{subject_index}_rep_{rep}.npy")

    subject_data = np.load(file_path, allow_pickle=True).item()
    predictions, uncertainties, explanations = subject_data['predictions'], subject_data['uncertainties'], subject_data['explanations']

        
    cfg = load_config()
    cfg.dataset.subject_index = subject_index
    _, _, _, _, _, _, _, ch_names = load_eeg_data(cfg)
    
    return predictions, uncertainties, explanations, ch_names

In [ ]:
def get_top_k_keys(d, k):
    """
    Returns the top k keys in a dictionary that have the highest values.

    Parameters:
    d (dict): The input dictionary.
    k (int): The number of top keys to return.

    Returns:
    list: A list of the top k keys with the highest values.
    """
    # Sort the dictionary by values in descending order and get the top k keys
    top_k_keys = sorted(d, key=d.get, reverse=True)[:k]
    return top_k_keys

# Example usage
d = {'a': 10, 'b': 20, 'c': 15, 'd': 5, 'e': 25}
k = 3
print(get_top_k_keys(d, k))  # Output: ['e', 'b', 'c']

In [ ]:
def load_all_subjects_topk():
    topk = np.load("top_k_abs.npy", allow_pickle=True).item()
    return topk

In [ ]:
def get_top_k_keys_all_subjects():
    topk = load_all_subjects_topk()
    topk_keys = {}
    for subject_index in topk.keys():
        topk_keys[subject_index] = get_top_k_keys(topk[subject_index], 10)
    return topk_keys


In [ ]:
top_k_all_subjects = get_top_k_keys_all_subjects()

In [ ]:
freq_bands = {
              "delta": (0, 4),
              "theta": (4, 8),
              "alpha": (8, 12),
              "beta": (12, 30),
              "gamma": (30, 45)}
phase_peturbations = np.arange(45, 316, 45)

In [ ]:
dir = "/home/marco/Documents/GitHub/tms_eeg_decoding/perturb_samples_phase/perturbed_predictions"

In [ ]:
import itertools

In [ ]:
cfg = load_config()
pairs = list(itertools.combinations(cfg.dataset.test_subject_indices, 2)) 

In [ ]:
def load_data_all_subjects():
    all_subjects_data = {}
    for subject_index in cfg.dataset.test_subject_indices:
        predictions, uncertainties, _, ch_names = load_predicted_amplitude_for_subject(subject_index)
        #top_k = load_subject_topk(subject_index)
        

        file_path = "/home/marco/Documents/GitHub/tms_eeg_decoding/data/subject_{:03d}_preprocessed_combined_py.fif".format(subject_index)
        epochs = mne.read_epochs(file_path)
        info_subj = epochs.info
        all_subjects_data[subject_index] = {"predictions": predictions, "uncertainties": uncertainties, "ch_names": ch_names, "info_subj": info_subj}
    return all_subjects_data

In [ ]:
all_subjects_data = load_data_all_subjects()

In [ ]:
def median_difference_all_subjects_dw(data_all_subjects, amplification_factors,take_abs=False, rep=1):
    distance_dir = "/home/marco/Documents/GitHub/tms_eeg_decoding/perturb_samples_phase/parallel_perturbation_distance_phase"
    pert_dir = "/home/marco/Documents/GitHub/tms_eeg_decoding/perturb_samples_phase/parallel_perturbation_phase"
    median_diff_per_channel_all_subjects = {}
    mean_diff_per_channel_all_subjects = {}
    std_diff_per_channel_all_subjects = {}
    for subject_index, data in data_all_subjects.items():
        predictions = data["predictions"]
        ch_names = data["ch_names"]
        median_diff_per_channel = {}
        mean_diff_per_channel = {}
        std_diff_per_channel = {}
        for band_name, (low_freq, high_freq) in freq_bands.items():
            median_diff_per_channel[band_name] = {}
            mean_diff_per_channel[band_name] = {}
            std_diff_per_channel[band_name] = {}
            for factor in amplification_factors:
            
                file_path_distance = file_path = f"parallel_distance_dict_{band_name}_channel_phase_shift_{factor}°_subject_{subject_index}_rep_{rep}.npy"
                load_path_distance = os.path.join(distance_dir, file_path_distance)
                distances = np.load(load_path_distance, allow_pickle=True).item()

                perturbed_data = np.load(f"{pert_dir}/parallel_perturbed_prediction_dict_{band_name}_channel_phase_shift_{factor}°_subject_{subject_index}_rep_{rep}.npy", allow_pickle=True).item()

                median_diff_per_channel[band_name][factor] = {}
                mean_diff_per_channel[band_name][factor] = {}
                std_diff_per_channel[band_name][factor] = {}
                for ch_name in ch_names:
                    perturbed_amplitude = perturbed_data[ch_name]
                    if take_abs:
                        diff = np.abs(predictions - perturbed_amplitude)/distances[ch_name]         
                    else:
                        diff = (predictions - perturbed_amplitude)/distances[ch_name]

                    median_diff_per_channel[band_name][factor][ch_name] = np.median(diff)
                    mean_diff_per_channel[band_name][factor][ch_name] = np.mean(diff)
                    std_diff_per_channel[band_name][factor][ch_name] = np.std(diff)
        median_diff_per_channel_all_subjects[subject_index] = median_diff_per_channel
        mean_diff_per_channel_all_subjects[subject_index] = mean_diff_per_channel
        std_diff_per_channel_all_subjects[subject_index] = std_diff_per_channel
    return median_diff_per_channel_all_subjects, mean_diff_per_channel_all_subjects, std_diff_per_channel_all_subjects

In [ ]:
median_diff_per_channel_all_subjects, median_diff_per_channel_all_subjects, std_diff_per_channel_all_subects = median_difference_all_subjects_dw(all_subjects_data, phase_peturbations, take_abs=True, rep=1)

In [ ]:
def plot_subject_channel_differences(subject_idx, amp_factor=2):
    """
    Plot the median difference per channel with standard deviation for a given subject and amplification factor.
    Sorts the importances in descending order in each subplot and color-codes by brain region.
    
    Parameters:
    -----------
    subject_idx : int
        The subject index to plot
    amp_factor : float
        The amplification factor to use (default: 2)
    """
    fig, axes = plt.subplots(len(freq_bands), 1, figsize=(15, 4*len(freq_bands)))
    
    # Get channel names for the subject
    ch_names = all_subjects_data[subject_idx]['ch_names']
    
    # Get top 10 channels for this subject from top_k_all_subjects
    top_channels = top_k_all_subjects.get(subject_idx, [])
    
    # Define brain region and hemisphere mapping
    brain_regions = {
        'F': 'Frontal', 'Fp': 'Frontopolar', 'AF': 'Anterior Frontal',
        'C': 'Central', 'T': 'Temporal', 'P': 'Parietal',
        'O': 'Occipital', 'PO': 'Parieto-occipital', 'FC': 'Fronto-central',
        'CP': 'Centro-parietal', 'TP': 'Temporo-parietal', 'FT': 'Fronto-temporal'
    }
    
    # Colors for different brain regions
    region_colors = {
        'Frontal': 'royalblue', 'Frontopolar': 'lightblue', 'Anterior Frontal': 'deepskyblue',
        'Central': 'green', 'Temporal': 'purple', 'Parietal': 'orangered',
        'Occipital': 'gold', 'Parieto-occipital': 'orange', 'Fronto-central': 'mediumseagreen',
        'Centro-parietal': 'tomato', 'Temporo-parietal': 'mediumpurple', 'Fronto-temporal': 'cornflowerblue'
    }
    
    # Plot each frequency band
    for i, (band_name, (low_freq, high_freq)) in enumerate(freq_bands.items()):
        # Get median and std differences for this band
        medians = []
        stds = []
        channels = []
        regions = []
        hemispheres = []
        
        for ch_name in ch_names:
            # Get median difference for this channel
            try:
                median_diff = median_diff_per_channel_all_subjects[subject_idx][band_name][amp_factor][ch_name]
                std_diff = std_diff_per_channel_all_subects[subject_idx][band_name][amp_factor][ch_name]
                medians.append(median_diff)
                stds.append(std_diff)
                channels.append(ch_name)
                
                # Determine brain region
                region_key = next((k for k in brain_regions.keys() if ch_name.startswith(k)), "Unknown")
                regions.append(brain_regions.get(region_key, "Unknown"))
                
                # Determine hemisphere (left, right, midline)
                if ch_name[-1].isdigit():
                    hemisphere = "Left" if int(ch_name[-1]) % 2 != 0 else "Right"
                else:
                    hemisphere = "Midline"
                hemispheres.append(hemisphere)
                
            except KeyError:
                continue
        
        # Sort channels by median difference in descending order
        sorted_indices = np.argsort(medians)[::-1]
        sorted_medians = [medians[idx] for idx in sorted_indices]
        sorted_stds = [stds[idx] for idx in sorted_indices]
        sorted_channels = [channels[idx] for idx in sorted_indices]
        sorted_regions = [regions[idx] for idx in sorted_indices]
        sorted_hemispheres = [hemispheres[idx] for idx in sorted_indices]
        
        # Create bar plot with error bars and color by brain region
        x = np.arange(len(sorted_channels))
        bars = axes[i].bar(x, sorted_medians, yerr=sorted_stds, alpha=0.7)
        
        # Color bars by brain region
        for j, bar in enumerate(bars):
            bar.set_color(region_colors.get(sorted_regions[j], 'gray'))
            
            # Add asterisk for top channels
            if sorted_channels[j] in top_channels:
                bar.set_edgecolor('red')
                bar.set_linewidth(2)
                axes[i].text(x[j], sorted_medians[j] + sorted_stds[j], '*', 
                             ha='center', va='bottom', fontsize=12, color='red')
        
        axes[i].set_title(f"{band_name} band ({low_freq}-{high_freq} Hz)")
        axes[i].set_ylabel("Median difference in prediction")
        
        # Set x-ticks and add hemisphere info to labels
        axes[i].set_xticks(x)
        labels = [f"{ch} ({hemi[0]})" for ch, hemi in zip(sorted_channels, sorted_hemispheres)]
        axes[i].set_xticklabels(labels, rotation=90)
    
    plt.tight_layout()
    plt.suptitle(f"Subject {subject_idx} - Amplification Factor {amp_factor}", fontsize=16)
    plt.subplots_adjust(top=0.95)
    
    # Add legend for brain regions
    handles = [plt.Rectangle((0,0),1,1, color=color) for color in region_colors.values()]
    labels = list(region_colors.keys())
    fig.legend(handles, labels, loc='upper right', bbox_to_anchor=(1, 0.98), ncol=2)
    
    # Add legend for hemisphere markers and top channels
    red_patch = plt.Rectangle((0,0),1,1, edgecolor='red', facecolor='none', linewidth=2)
    fig.legend([red_patch], ['Top 10 important channels'], 
               loc='upper right', bbox_to_anchor=(1, 0.92))


In [ ]:
plot_subject_channel_differences(2, amp_factor=180)

In [ ]:
def get_top_k_keys(dictionary, k):
    """
    Returns the keys corresponding to the k highest values in the dictionary.
    """
    sorted_items = sorted(dictionary.items(), key=lambda x: x[1], reverse=True)
    top_k_keys = [item[0] for item in sorted_items[:k]]
    return top_k_keys

def analyze_top_channel_importance(amp_factor = 2):
    """
    Analyze how much more important the top 5 channels of each frequency band are
    compared to the mean of all channels not in top 5 in that frequency band.
    """
    results = {}
    
    for subject_idx in cfg.dataset.test_subject_indices:
        results[subject_idx] = {}
        
        # Get channel names for the subject
        ch_names = all_subjects_data[subject_idx]['ch_names']
        
        for band_name in freq_bands.keys():
            # Use amplification factor of 2
            
            
            # Get median differences for all channels in this band
            channel_importances = {}
            for ch_name in ch_names:
                try:
                    median_diff = median_diff_per_channel_all_subjects[subject_idx][band_name][amp_factor][ch_name]
                    channel_importances[ch_name] = median_diff
                except KeyError:
                    continue
            
            if not channel_importances:
                continue
            
            # Get top 5 channels for this band
            top_channels = get_top_k_keys(channel_importances, 5)
            
            # Calculate mean importance of top 5 channels
            top_5_values = [channel_importances[ch] for ch in top_channels]
            top_5_mean = np.mean(top_5_values) if top_5_values else 0
            
            # Calculate mean importance of channels NOT in top 5
            non_top_channels = [ch for ch in channel_importances if ch not in top_channels]
            non_top_values = [channel_importances[ch] for ch in non_top_channels]
            non_top_mean = np.mean(non_top_values) if non_top_values else 0
            
            # Calculate ratio of top 5 mean to non-top 5 channels mean
            importance_ratio = top_5_mean / (non_top_mean ) if non_top_mean != 0 else 0
            
            results[subject_idx][band_name] = {
                'top_5_channels': top_channels,
                'top_5_mean': top_5_mean,
                'non_top_mean': non_top_mean,
                'importance_ratio': importance_ratio
            }
    
    return results

def visualize_importance_analysis(importance_results):
    """
    Visualize the results of the importance analysis with two plots:
    1. Importance ratios across subjects for each frequency band
    2. Most common top 5 channels for each frequency band
    """
    # Setup for plotting ratios
    fig, axes = plt.subplots(len(freq_bands), 1, figsize=(15, 4*len(freq_bands)))
    
    bands = list(freq_bands.keys())
    subjects = sorted(list(importance_results.keys()))
    
    # Plot importance ratios
    for i, band in enumerate(bands):
        ratios = [importance_results[subj][band]['importance_ratio'] for subj in subjects if band in importance_results[subj]]
        subj_indices = [subj for subj in subjects if band in importance_results[subj]]
        
        axes[i].bar(subj_indices, ratios)
        axes[i].set_title(f"{band} band ({freq_bands[band][0]}-{freq_bands[band][1]} Hz): Top 5 / Non-top channels importance ratio")
        axes[i].set_xlabel("Subject")
        axes[i].set_ylabel("Ratio: Top 5 / Non-top channels")
        axes[i].grid(axis='y', linestyle='--', alpha=0.7)
        
        # Add mean line
        mean_ratio = np.mean(ratios)
        axes[i].axhline(y=mean_ratio, color='r', linestyle='-', label=f'Mean ratio: {mean_ratio:.2f}')
        axes[i].legend()
    
    plt.tight_layout()
    plt.figure(figsize=(15, 20))
    
    # Collect common top channels
    common_channels = {band: {} for band in bands}
    
    for subject_idx in importance_results:
        for band in bands:
            if band not in importance_results[subject_idx]:
                continue
            top_channels = importance_results[subject_idx][band]['top_5_channels']
            
            for channel in top_channels:
                if channel in common_channels[band]:
                    common_channels[band][channel] += 1
                else:
                    common_channels[band][channel] = 1
    
    # Plot common channels
    for i, band in enumerate(bands):
        plt.subplot(len(bands), 1, i+1)
        
        # Sort by frequency
        sorted_channels = sorted(common_channels[band].items(), key=lambda x: x[1], reverse=True)
        top_10_channels = sorted_channels[:10]
        
        channels = [ch[0] for ch in top_10_channels]
        counts = [ch[1] for ch in top_10_channels]
        
        # Create horizontal bar chart
        bars = plt.barh(channels, counts)
        plt.title(f"Most common top 5 channels for {band} band ({freq_bands[band][0]}-{freq_bands[band][1]} Hz)")
        plt.xlabel("Number of subjects")
        
        # Add count labels
        for bar in bars:
            width = bar.get_width()
            plt.text(width + 0.3, bar.get_y() + bar.get_height()/2, 
                    f'{width:.0f}', ha='left', va='center')
    
    plt.tight_layout()


# Run the analysis
importance_results = analyze_top_channel_importance(amp_factor=180)
#visualize_importance_analysis(importance_results)

In [ ]:
def plot_importance_ratios_by_frequency_band():
    """
    Create a plot showing importance ratios for each subject across different frequency bands.
    Each frequency band will be displayed in a separate subplot.
    Also adds an aggregated plot comparing frequency bands.
    """
    # Create figure with subplots for each frequency band
    fig, axes = plt.subplots(len(freq_bands), 1, figsize=(15, 4*len(freq_bands)))
    
    # Get list of subjects and bands
    subjects = sorted(list(importance_results.keys()))
    bands = list(freq_bands.keys())
    
    # Store band statistics for aggregated plot
    band_stats = {band: [] for band in bands}
    
    # Define a colormap for better visualization
    colors = plt.cm.viridis(np.linspace(0, 0.8, len(freq_bands)))
    x = np.arange(len(subjects))

    # Plot importance ratios for each frequency band
    for i, band in enumerate(bands):
        # Extract importance ratios for this band across all subjects
        valid_subjects = [subj for subj in subjects if band in importance_results.get(subj, {})]
        ratios = [importance_results[subj][band]['importance_ratio'] for subj in valid_subjects]
        
        # Store ratios for aggregated plot
        band_stats[band] = ratios
        
        # Create bar plot
        bars = axes[i].bar(x, ratios, color=colors[i], alpha=0.7)
        
        # Add mean line
        mean_ratio = np.mean(ratios)
        axes[i].axhline(y=mean_ratio, color='r', linestyle='-', 
                        label=f'Mean ratio: {mean_ratio:.2f}')
        
        # Add labels and formatting
        axes[i].set_title(f"{band.capitalize()} band ({freq_bands[band][0]}-{freq_bands[band][1]} Hz)", 
                         fontsize=14, fontweight='bold')
        axes[i].set_xlabel("Subject ID", fontsize=12)
        axes[i].set_ylabel("Top 5 / Non-top Importance Ratio", fontsize=12)
        axes[i].grid(axis='y', linestyle='--', alpha=0.7)
        axes[i].legend(loc='upper right')
        axes[i].set_xticks(x)
        axes[i].set_xticklabels(cfg.dataset.test_subject_indices, rotation=45)
    
    plt.tight_layout()
    plt.suptitle("Importance Ratio by Frequency Band: Top 5 Channels vs. Other Channels", 
                 fontsize=16, fontweight='bold', y=1.02)
    
    # Create aggregated plot comparing frequency bands
    plt.figure(figsize=(12, 8))
    
    # Calculate statistics for each band
    means = [np.mean(band_stats[band]) for band in bands]
    medians = [np.median(band_stats[band]) for band in bands]
    stds = [np.std(band_stats[band]) for band in bands]
    
    # Create x positions for bars
    x_pos = np.arange(len(bands))
    width = 0.35
    
    # Plot bars for mean and median
    plt.bar(x_pos - width/2, means, width, label='Mean Ratio', color='steelblue', alpha=0.7)
    plt.bar(x_pos + width/2, medians, width, label='Median Ratio', color='lightcoral', alpha=0.7)
    
    # Add error bars for standard deviation
    plt.errorbar(x_pos - width/2, means, yerr=stds, fmt='none', color='darkblue', capsize=5)
    
    # Add labels and formatting
    plt.xlabel('Frequency Band', fontsize=14)
    plt.ylabel('Importance Ratio (Top 5 / Non-top)', fontsize=14)
    plt.title('Aggregated Importance Ratios by Frequency Band', fontsize=16, fontweight='bold')
    plt.xticks(x_pos, [f"{band}\n({freq_bands[band][0]}-{freq_bands[band][1]} Hz)" for band in bands])
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    plt.legend()
    
    # Add text with exact values
    for i, (mean, median, std) in enumerate(zip(means, medians, stds)):
        plt.text(i - width/2, mean + 0.1, f'{mean:.2f}±{std:.2f}', 
                 ha='center', va='bottom', fontsize=10, rotation=0)
        plt.text(i + width/2, median + 0.1, f'{median:.2f}', 
                 ha='center', va='bottom', fontsize=10, rotation=0)
    
    plt.tight_layout()

# Create and display the plot
plot_importance_ratios_by_frequency_band()

For the upper frequency band the importanec of the most important channels compared to the other channels seems to be higher

In [ ]:
median_diff_per_channel_all_subjects, median_diff_per_channel_all_subjects, std_diff_per_channel_all_subects = median_difference_all_subjects_dw(all_subjects_data, phase_peturbations, take_abs=False, rep=1)

In [ ]:
import matplotlib.ticker as ticker
def plot_sum_topomap(median_diff_per_channel_all_subjects, amp_factor=180, axes=None, idx=0):
    if axes is None:
        fig,axes = plt.subplots(1, 5, figsize=(16, 5))
    
    for i, (band_name, (low_freq, high_freq)) in enumerate(freq_bands.items()):
        channel_importances = np.zeros(len(median_diff_per_channel_all_subjects[1]["delta"][amp_factor].keys()))
        for subject_index in median_diff_per_channel_all_subjects.keys():
            median_diff_per_channel = list(median_diff_per_channel_all_subjects[subject_index][band_name][amp_factor].values())
            channel_importances += median_diff_per_channel
     
        topo,_ = mne.viz.plot_topomap(channel_importances, all_subjects_data[1]['info_subj'], axes=axes[i], show=False)
        if idx==0:
            axes[i].set_title(f"{band_name} band", fontsize=18)
        cb = plt.colorbar(topo, ax=axes[i], location='bottom', pad=0.05, shrink=0.8)
        cb.set_label('Median $\Delta p_w$', fontsize=14)
        cb.ax.tick_params(labelsize=14)
        cb.ax.yaxis.set_major_formatter(ticker.FormatStrFormatter('%.2f'))
        
    plt.savefig(f"sum_topomap_phase_phase_shift_{amp_factor}.png")


In [ ]:
median_diff_per_channel_all_subjects, median_diff_per_channel_all_subjects, std_diff_per_channel_all_subects = median_difference_all_subjects_dw(all_subjects_data, phase_peturbations,take_abs=False)

In [ ]:
for phase_shift in phase_peturbations:
    plot_sum_topomap(median_diff_per_channel_all_subjects, amp_factor=phase_shift)

In [ ]:
fig, axs = plt.subplots(3, 5, figsize=(16, 12))

fig.tight_layout(rect=[0, 0, 1, 0.95])
fig.subplots_adjust(wspace=-0.1)
for idx,phase_shift in enumerate([45, 180, 315]):
    plot_sum_topomap(median_diff_per_channel_all_subjects, amp_factor=phase_shift, axes=axs[idx], idx=idx)